# Convolution Tutorial Series — Part 4: Groups

In [ ]:
from IPython.display import Video

video_path = '/kaggle/input/datasets/massimilianoghiotto/neurogolf-convseries-part2-supportvideo/PaddingTutorial.mp4'

Video(video_path, width=750, height=450, embed=True)

**Goal:** Understand the **group** attribute: how to cut parameters by splitting channels into independent processing groups. This is the fourth episode in our series covering the ONNX Conv function. We will explain how to optimize your model's parameter footprint using ARC Task 258. The previous episodes covered [Kernel Size & Bias](https://www.kaggle.com/code/massimilianoghiotto/convolution-series-part-1), [Padding](https://www.kaggle.com/code/massimilianoghiotto/convolution-series-part-2), and [Stride](https://www.kaggle.com/code/massimilianoghiotto/convolution-series-part-3).

**Task (ARC 258):**
1. **Recap:** Red fills gaps of exactly 1 between two Blues. 
2. **The Problem:** A standard [10, 10, 1, 3] convolution has a huge parameter cost because every output color looks at every input color.
3. **The Solution:** Use **groups** to isolate the active colors (Black, Blue, Red) and ignore the unused ones.

**Score formula:** `Points = max(1.0, 25.0 - log(Mem_bytes + Params))`

**Our result:** Mem=0, Params=160, **19.925 points** (Up from 19.263 pts with group=1)

---

In [ ]:
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null

In [ ]:
import json, warnings, os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

arc_colors = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00', 
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBCA', '#870C25'
]
color_names = ['black','blue','red','green','yellow','gray','magenta','orange','teal','maroon']

def plot_arc_grid(grid, ax, title=''):
    H, W = len(grid), len(grid[0])
    img = np.zeros((H, W, 3), dtype=np.uint8)
    for r in range(H):
        for c in range(W):
            hex_c = arc_colors[grid[r][c]].lstrip('#')
            img[r,c] = [int(hex_c[i:i+2], 16) for i in (0, 2, 4)]
    ax.imshow(img, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title, fontsize=10, fontweight='bold')
    for r in range(H+1): ax.axhline(r-0.5, color='gray', lw=0.5, alpha=0.3)
    for c in range(W+1): ax.axvline(c-0.5, color='gray', lw=0.5, alpha=0.3)

with open('/kaggle/input/competitions/neurogolf-2026/task258.json') as f:
    task = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_arc_grid(task['train'][0]['input'], axes[0], 'Input')
plot_arc_grid(task['train'][0]['output'], axes[1], 'Target Output')
plt.show()

## 1. The Parameter Problem

By default, we deal with 10 channels (one per color). With the default `group=1`, every output channel is connected to **all 10 input channels**.

For our 1x3 kernel in Task 258:
$$\text{Weight shape} = [O, I, k_H, k_W] = [10, 10, 1, 3]$$
That is $10 \times 10 \times 1 \times 3 = 300$ weights + 10 biases = **310 parameters**.

But look at the input: only **Black (0)** and **Blue (1)** are active. The other 8 colors are just empty channels. The output is the same, with the addition of the **Red (2)** color. 

**Dead Weight:** In a standard convolution, we end up storing hundreds of zeros for colors that don't exist in the task. If the sparsity pattern of the convolution has a particular structure we can get rid of some of the, and boost our score, without changing the logic!

## 2. How Groups Work

The `group` attribute divides input channels into **G independent groups**. Each output channel only "sees" the input channels in its own group.

- **`group=1` (Default):** Every output see every input, i.e., output 0 sees inputs [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
- **`group=2`:** Output [0-4] sees Inputs [0-4]; Output [5-9] sees Inputs [5-9].
- **`group=5`:** Output [0-1] sees Inputs [0-1]; Output [2-3] sees Inputs [2-3], and so on.
- **`group=10` (Depthwise):** Output 0 sees Input 0, Output 1 sees Input 1, etc.

### The Math
When `group=G`, the weight shape becomes: `[O, I/G, k_H, k_W]`.

For Task 258 with `group=2`:
$$\text{Shape} = [10, 10/2, 1, 3] = [10, 5, 1, 3]$$
This cuts our weights from 300 down to **150**! 

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5))

for ax_i, (title, G) in enumerate([
    ('group=1 (Full Interaction)', 1),
    ('group=2 (Isolated Groups)', 2),
    ('group=2 (Isolated Groups)', 5),
    ('group=10 (Depthwise)', 10),
]):
    ax = axes[ax_i]
    C_per_G = 10 // G
    img = np.ones((10, 10, 3))
    for out_ch in range(10):
        group_idx = out_ch // (10 // G)
        in_start = group_idx * C_per_G
        in_end = in_start + C_per_G
        img[out_ch, in_start:in_end] = [0, 0.46, 0.85]
    ax.imshow(img, interpolation='nearest')
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xlabel('Input Channel'); ax.set_ylabel('Output Channel')
    ax.set_title(title, fontsize=11, fontweight='bold')
    for r in range(11): ax.axhline(r-0.5, color='gray', lw=0.5)
    for c in range(11): ax.axvline(c-0.5, color='gray', lw=0.5)
    if G > 1:
        for g in range(1, G):
            ax.axhline(g * C_per_G - 0.5, color='red', lw=2, ls='--')
            ax.axvline(g * C_per_G - 0.5, color='red', lw=2, ls='--')
    ax.text(0.5, -0.25, f'Weight Shape: [10, {C_per_G}, 1, 3]', transform=ax.transAxes, ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Optimizing Task 258

Our rule requires interactions between **Black (0)** and **Blue (1)** to create **Red (2)**. 
- With `group=2`, channels 0, 1, and 2 all fall into **Group 0** (which covers channels 0-4).
- Group 0 output channels can see Group 0 input channels. Perfect!

| Output Channel | Can See Input | Role |
| :--- | :--- | :--- |
| **0 (Black)** | [0, 1, 2, 3, 4] | Identity + Negative Blue neighbors |
| **1 (Blue)** | [0, 1, 2, 3, 4] | Identity |
| **2 (Red)** | [0, 1, 2, 3, 4] | Sum Blue neighbors |
| **5-9 (Unused)** | [5, 6, 7, 8, 9] | Always zero (input is black) |

By choosing `group=2`, we effectively throw away the connections between the first 5 colors and the last 5 colors, saving 150 parameters.

In [ ]:
def create_task258_optimized():
    # M=10, C/G=5, kH=1, kW=3
    W = np.zeros((10, 5, 1, 3), dtype=np.float32)
    
    # Output 0 (Black): stays Black if Input 0 (Black) is center, 
    # but Blue neighbors (Input 1) subtract score
    W[0, 0, 0, 1] = 4.0
    W[0, 1, 0, 0] = -2.0
    W[0, 1, 0, 2] = -2.0
    
    # Output 1 (Blue): Identity from Input 1 (Blue)
    W[1, 1, 0, 1] = 2.0
    
    # Output 2 (Red): Sum neighbors from Input 1 (Blue)
    W[2, 1, 0, 0] = 2.0
    W[2, 1, 0, 2] = 2.0

    # Biases as thresholds
    B = np.array([-1.0, -1.0, -3.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], dtype=np.float32)

    X = helper.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
    Y = helper.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])

    node = helper.make_node("Conv", ["input", "W", "B"], ["output"],
                            kernel_shape=[1, 3], pads=[0, 1, 0, 1], group=2)

    graph = helper.make_graph([node], "optimized_groups", [X], [Y], initializer=[
        numpy_helper.from_array(W, name="W"),
        numpy_helper.from_array(B, name="B"),
    ])
    
    model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", 11)])
    return model

model = create_task258_optimized()
onnx.save(model, "task258_optimized.onnx")

In [ ]:
def run_inference(grid, model_path):
    oh = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            oh[0, grid[r][c], r, c] = 1.0
    sess = ort.InferenceSession(model_path)
    out = sess.run(None, {'input': oh})[0]
    return np.argmax(out[0], axis=0)

inp = task['train'][0]['input']
target = task['train'][0]['output']
pred = run_inference(inp, "task258_optimized.onnx")

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
plot_arc_grid(inp, axes[0], 'Input')
plot_arc_grid(target, axes[1], 'Target')
plot_arc_grid(pred[:len(inp), :len(inp[0])], axes[2], 'Optimized Pred')
plt.show()

# Verify with official checker
import sys
sys.path.append("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
from neurogolf_utils import *
passed = verify_network(model, 258, task)

## 4. Parameter Comparison Table

| group | Weight Shape | Weights | Biases | Total Params | Savings |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1 (Default)** | [10, 10, 1, 3] | 300 | 10 | 310 | — |
| **2 (Our choice)** | [10, 5, 1, 3] | 150 | 10 | **160** | **48%** |
| **5** | [10, 2, 1, 3] | 60 | 10 | 70 | 77% |
| **10 (Depthwise)** | [10, 1, 1, 3] | 30 | 10 | 40 | 87% |

**Note:** `group=10` is very efficient but only works if an output color *only* depends on its own input color (no cross-color interactions). Since our Red (2) depends on Blue (1), `group=10` would fail here, but `group=2` works perfectly!

# Summary
- **`group=G`** breaks the 10x10 interaction matrix into G smaller, independent blocks.
- **Efficiency:** It dramatically reduces the parameter count (and your NeuroGolf score).
- **Isolation:** Only colors within the same group can interact.

In [ ]:
import shutil
import os
import zipfile

# --- CONFIGURATION ---
SOURCE_FOLDER = '/kaggle/input/datasets/massimilianoghiotto/neurogolf2026-6255/submission'
OUTPUT_ZIP = '/kaggle/working/submission.zip'

# Package the ZIP (Ensuring files are at the root)
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(SOURCE_FOLDER):
        for file in files:
            if file.endswith('.onnx'):
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, SOURCE_FOLDER))

# Acknowledgments
Big shoutout to everyone who shared ideas and notebooks 
@cdeotte, @kojimar, @konbu17, @mirzayasirabdullah07, @nadeembinshajahan, @octaviograu, @rauffauzanrambe, @seddiktrk, @yash9439 and many more that I might have missed.

### **This is the best that we are able to do for the moment, if anyone has any suggestion, please write it in the comments, we are happy to have some brainstorning between people.**